# SGLang

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models. It provides an OpenAI-compatible HTTP server that LlamaIndex connects to via the `llama-index-llms-sglang` integration.

SGLang supports multiple hardware backends:
- **CUDA** (NVIDIA GPUs)
- **XPU** (Intel GPUs — Arc, Flex, Data Center GPU)
- **CPU** (fallback)

This notebook covers:
1. Installing dependencies and starting the SGLang server
2. Basic completion
3. Chat
4. Streaming
5. Async
6. Running on Intel XPU

## 1. Installation

In [ ]:
%pip install llama-index-llms-sglang

Note: you may need to restart the kernel to use updated packages.


## 2. Start the SGLang Server

Before running any of the cells below, start the SGLang server in a separate terminal.

**CUDA (NVIDIA GPU):**
```bash
pip install sglang[all]
python -m sglang.launch_server \
    --model-path mistralai/Mistral-7B-Instruct-v0.1 \
    --port 30000
```

**XPU (Intel GPU) via Docker:**

```bash
# Clone the SGLang repo (the Dockerfile lives there)
git clone https://github.com/sgl-project/sglang.git
cd sglang
git checkout ddaf430e6c59a88da0a6cca4c71033cedf102a88

# Build the XPU image
docker build -t sglang-xpu:latest -f docker/xpu.Dockerfile .

# with proxy
docker build -t sglang-xpu:latest -f docker/xpu.Dockerfile   --build-arg http_proxy="${http_proxy}"   --build-arg https_proxy="${https_proxy}"   --build-arg no_proxy="${no_proxy}"   --build-arg ftp_proxy="${ftp_proxy}"  .

# Launch the server
# If behind a corporate proxy, add -e http_proxy=... -e https_proxy=... to the docker run command
docker run \
  -it --rm --privileged \
  --ipc=host \
  --network=host \
  --user root \
  --group-add "$(getent group video | cut -d: -f3)" \
  --device /dev/dri \
  -v /dev/dri/by-path:/dev/dri/by-path \
  -v /dev/shm:/dev/shm \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e ZE_AFFINITY_MASK=0 \
  -e ONEAPI_DEVICE_SELECTOR=level_zero:0 \
  -e http_proxy="${http_proxy}" \
  -e https_proxy="${https_proxy}" \
  -e ftp_proxy="${ftp_proxy}" \
  -e no_proxy="${no_proxy}" \
  sglang-xpu:latest \
  /bin/bash -c 'source /opt/intel/oneapi/setvars.sh --force && sglang serve \
    --model-path Qwen/Qwen2.5-1.5B-Instruct \
    --trust-remote-code \
    --disable-overlap-schedule \
    --device xpu \
    --host 0.0.0.0 \
    --attention-backend intel_xpu \
    --page-size 128 \
    --tool-call-parser qwen \
    --grammar-backend xgrammar \
    --mem-fraction-static 0.7'
```

Key flags:
- `--network=host`: exposes port 30000 directly on the host
- `ZE_AFFINITY_MASK=0,1`: selects XPU tile/device indices
- `ONEAPI_DEVICE_SELECTOR=level_zero:*`: forces Level Zero backend for Intel GPU
- `--tp 2`: tensor parallelism across 2 XPU devices
- `--attention-backend intel_xpu`: Intel-optimized attention kernel

Wait until you see `Server is ready` before proceeding.

## 3. Basic Setup

In [ ]:
import sys

sys.path.insert(0, "/home/dut3806/sy/sglang")
import os

os.environ["no_proxy"] = "localhost,127.0.0.1"
from llama_index.llms.sglang import SGLang
from llama_index.core.llms import ChatMessage

llm = SGLang(
    model="Qwen/Qwen3-4B-Instruct-2507",
    api_url="http://localhost:30000",
    temperature=0.7,
    max_new_tokens=256,
    is_chat_model=True,
)

## 4. Completion

In [ ]:
response = llm.complete("What is a black hole?")
print(response)

A black hole is a region in space where the gravitational pull is so strong that nothing, not even light, can escape from it. It's named "black" because it doesn't emit light of its own, and instead absorbs all incoming radiation.

Here are some key points about black holes:

1. Formation: Black holes form when massive stars collapse at the end of their life cycle, creating an incredibly dense object with gravity so strong that nothing (not even photons) can escape from it.

2. Types:
   - Stellar-mass black hole: Forms from the collapse of a single star.
   - Supermassive black hole: Found at the center of most galaxies, including our Milky Way.
   - Microquasar: An intermediate-sized black hole with mass between solar masses to millions of solar masses.

3. Event Horizon: The boundary around a black hole beyond which nothing can be seen or interacted with by anything outside. This area is called the event horizon.

4. Spacetime curvature: As matter falls into a black hole, the fabric

## 5. Chat

In [ ]:
messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(
        role="user", content="Explain quantum entanglement in simple terms."
    ),
]
response = llm.chat(messages)
print(response)

assistant: Quantum entanglement is a phenomenon in quantum mechanics where pairs or groups of particles become interconnected and the state of one particle cannot be described independently of the state of another, no matter how far apart they are. In simple terms, it's like being able to instantly know something about two separate objects that aren't physically connected.

Imagine you have two magical dice, each with faces numbered from 1 to 6. If you roll them simultaneously, they end up showing different numbers (for example, Dice A shows a 4 and Dice B shows an 8). When measuring the number on each die, you can predict what number the other die will show just by looking at its own result without actually touching or interacting with it directly. This ability to instantly share information between distant parts of the system defies classical physics rules.

This instantaneous connection describes quantum entanglement. It demonstrates the strange interwoven nature of quantum world ph

## 6. Streaming

In [ ]:
# Streaming completion
for chunk in llm.stream_complete("Tell me a short story about a robot."):
    print(chunk.delta, end="", flush=True)

Once upon a time, there was a robot named Robby. He had been designed to assist humans and make their lives easier. Every day he went about his duties with precision and efficiency.
But one day, as Robby worked in the kitchen preparing a meal for a human customer, he noticed something unusual. The oven kept shutting off unexpectedly, which made it difficult for him to complete his tasks on time.
Robby knew that something needed to be done quickly to prevent any further issues. So, he spent hours researching and experimenting until he discovered how to fix the problem himself - by adding an extra step to the cooking process!
From that day on, Robby became even more efficient than before because of his new-found knowledge. But despite this success, he never forgot what motivated him to create such a device: to help people live better lives.
Even though he could do everything without needing human interaction anymore, Robby always felt grateful to have been created to serve humanity. And 

In [ ]:
# Streaming chat
messages = [ChatMessage(role="user", content="What is the speed of light?")]
for chunk in llm.stream_chat(messages):
    print(chunk.delta, end="", flush=True)

The speed of light in a vacuum is approximately 299,792 kilometers per second (km/s) or about 186,282 miles per second. This constant, known as the speed of light in a vacuum, represents the maximum speed at which all energy, matter, and information can travel through space.

## 7. Async

In [ ]:
response = await llm.acomplete("What is the meaning of life?")
print(response)

The meaning of life is a philosophical question that has puzzled humans for centuries. It's difficult to define because it can be subjective and vary from person to person.
However, many people believe that the purpose of life is to seek happiness and fulfillment in one's experiences. Others may prioritize relationships with loved ones or finding ways to make a positive impact on the world around them.
Ultimately, there is no single answer to this question, as different individuals will have their own unique values and beliefs when it comes to what constitutes a meaningful life.


In [ ]:
messages = [ChatMessage(role="user", content="What is the meaning of life?")]
response = await llm.achat(messages)
print(response)

assistant: As an AI language model, I don't have personal beliefs or opinions. However, the meaning of life is a complex and subjective topic that has been debated by philosophers, scientists, and individuals for centuries.

Some people believe that the meaning of life is to seek happiness, while others argue that it's about finding purpose in one's work or relationships with others. Some may find value in exploring their passions, making a positive impact on society, or achieving inner peace and self-fulfillment.

Ultimately, what defines the meaning of life can vary greatly depending on individual values, experiences, and perspectives.


In [ ]:
# Async streaming completion
async for chunk in await llm.astream_complete("Describe the solar system."):
    print(chunk.delta, end="", flush=True)

The solar system is the collection of celestial bodies that orbit around a common barycenter, primarily consisting of the Sun and eight planets. Here are the essential components:

1. **Sun**: The central star, from which the entire solar system derives its energy.

2. **Planets**:
   - Eight major planets: Mercury, Venus, Earth/Moon, Mars, Jupiter, Saturn, Uranus, Neptune.
   - Several dwarf planets: Pluto, Eris, Haumea, Makemake, Ceres.
   - Several small bodies like asteroids and comets.

3. **Moons**: Natural satellites that orbit one or more planets. Examples include Moon for Earth, Io for Jupiter, etc.

4. **Orbiting Bodies**:
   - Asteroids (mainly smaller than moons but larger than many other objects in space).
   - Trans-Neptunian Objects (TNOs) located beyond Neptune's orbit.
   - Kuiper Belt Objects (KBOs), including Pluto and similar bodies.
   - Centaurs, trans-asteroid belt objects that also travel between the asteroid belt and outer planets.

5. **Space Debris**: Objects

In [ ]:
# Async streaming chat
messages = [ChatMessage(role="user", content="Describe the solar system.")]
async for chunk in await llm.astream_chat(messages):
    print(chunk.delta, end="", flush=True)

The solar system is the gravitational assembly of the Sun and all objects gravitationally bound to it, including planets, dwarf planets, comets, asteroids, meteoroids, and other small Solar System bodies. It formed approximately 4.6 billion years ago from a nebular disk around our star.

At its center lies the Sun, which makes up about 99.8% of the system's mass due to its immense size and energy output. The remaining matter forms an orbiting planetary system composed of eight major planets, five dwarf planets, numerous moons, asteroids, and other smaller celestial bodies. These planets are classified as follows:

1. **Terrestrial Planets**: Mercury, Venus, Earth, and Mars
2. **Jovian Planets**: Jupiter, Saturn, Uranus, and Neptune

These planets vary in their composition, ranging from rocky like Earth and Mars to gas giants like Jupiter and Saturn. They are orbited by natural satellites (moons) ranging from tiny Ganymede (the largest moon in the solar system) to Titan (Saturn’s larges

## 8. Running on Intel XPU

SGLang supports Intel GPUs (Arc, Flex, Data Center GPU) via the `--device xpu` flag when launching the server. The LlamaIndex client requires no changes — it connects to the same HTTP endpoint regardless of the backend device.

To verify the server is using XPU, check the server startup logs for:
```
Device: xpu
```

Once the XPU server is running on port 30000, use the same `SGLang` client as above:

In [ ]:
# Same client code works regardless of whether server runs on CUDA or XPU
llm_xpu = SGLang(
    model="Qwen/Qwen3-4B-Instruct-2507",
    api_url="http://localhost:30000",
    temperature=0.7,
    max_new_tokens=256,
    is_chat_model=True,
)

response = llm_xpu.complete("What is a black hole?")
print(response)

A black hole is an astronomical object with a gravitational field so strong that nothing, not even light, can escape from it. It is formed when a massive star collapses under its own gravity at the end of its life cycle.

Here are some key characteristics and processes related to black holes:

1. Formation: Black holes typically form after a massive star runs out of nuclear fuel in its core. When a star's core reaches a certain size (around 3 million times the mass of our Sun), the immense pressure inside causes electrons and protons to combine into neutrons. This process creates a neutron star or if more than 2-3 solar masses are involved, it results in a supernova explosion, leaving behind a black hole.

2. Event Horizon: The boundary around a black hole beyond which anything including light cannot escape is called the event horizon. It's essentially a one-dimensional surface with three dimensions that account for time within the event horizon but none outside it due to the extreme e

## 9. LLM Metadata

In [ ]:
print(llm.metadata)

context_window=3900 num_output=256 is_chat_model=True is_function_calling_model=False model_name='Qwen/Qwen3-4B-Instruct-2507' system_role=<MessageRole.SYSTEM: 'system'>
